# 🚕 Diagnostic Lab: NYC Yellow Taxi
**Exploratory Data Analysis (EDA) and Quality Audit**

## 1. The Expectation (Initial Ingestion)
In this lab, we act as data auditors. The official dataset from New York's Taxi & Limousine Commission (TLC) is one of the richest open datasets in the world, containing GPS telemetry and detailed financial billing for every yellow taxi trip in the city.

Our expectation within the Medallion architecture is to ingest raw data (Bronze Layer) and produce high-reliability executive Data Marts (Gold Layer). To achieve this, we will leverage **DuckDB** to directly explore the Parquet file without overloading RAM, performing native statistical processing.

In [ ]:
import duckdb
import plotly.express as px
import os
import requests

# Establishing in-memory connection with DuckDB
con = duckdb.connect(':memory:')

URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-04.parquet"

# Raw file path for the Bronze Layer
RAW_FILE = "../data/raw/yellow_tripdata_2026-04.parquet"

print(f"DuckDB Engine {duckdb.__version__} initialized and ready to explore the Bronze Layer.")

In [ ]:
# Downloads the dataset only if it doesn't already exist
if not os.path.exists(RAW_FILE):
    print("Downloading dataset...")
    response = requests.get(URL)
    with open(RAW_FILE, "wb") as f:
        f.write(response.content)
    print("Download completed!")
else:
    print("File already exists in data/ directory.")

## 2. Statistical Diagnosis
Before plotting charts or computing business KPIs, we need to assess structural data health. Real-world urban IoT and financial transaction environments are chaotic. Hardware failures, GPS shadow zones (common in Manhattan's skyscraper canyons), and systemic cancellations can corrupt the dataset.

We will use DuckDB's native `SUMMARIZE` command to generate a complete statistical diagnostic of the file in milliseconds.

In [ ]:
# SUMMARIZE natively computes min, max, nulls, distinct counts, and descriptive statistics
query_summarize = f"""
    SUMMARIZE SELECT * FROM '{RAW_FILE}'
"""

# Fetching the result directly into a DataFrame for tabular visualization
df_summary = con.sql(query_summarize).df()
df_summary[['column_name', 'column_type', 'min', 'max', 'null_percentage']]

### ⚠️ Critical Findings

Observing the statistical summary generated by SUMMARIZE, we identified severe physical and systemic anomalies that render raw data unusable for executive BI reporting. The raw dataset is corrupted across the following pillars:

1. **Financial Collapse (Negative Values)**: The minimum value of nearly all financial columns is heavily negative (e.g., total_amount reaches -1278.4 and tip_amount -222.0). This proves the TLC system lacks an isolated "Trip Status" column; refunds, credit card fraud, or cancellations are registered by mirroring billing negatively, which would distort company P&L reports.

2. **Hardware Anomalies (Uncalibrated GPS & Clock)**: Physical telemetry is severely compromised. The trip_distance column registers an absurd maximum value of 281,576.08 miles (greater than the distance from Earth to the Moon) and a minimum of 0.0, highlighting GPS "shadowing" in urban canyons. Furthermore, there are "time travels": timestamps (tpep_pickup_datetime) starting in 2001-01-01, indicating taximeters with unconfigured RTC clocks or dead batteries.

3. **Block Systemic Failure (20.88% Data Gap)**: Missing data is not organic or random. Exactly 20.88% of the dataset simultaneously lost passenger_count, RatecodeID, store_and_fwd_flag, congestion_surcharge, and Airport_fee. This block behavior points to a severe technical integration failure, likely caused by an API outage from a specific technology provider (VendorID).

4. **Ghost Operations**: The minimum value of passenger_count is 0, indicating taximeters running and charging without passengers on board (potentially signaling parcel delivery, operational fraud, or driver input failure).

## 3. Visual Evidence (Plotly + Native DuckDB)
To avoid freezing the Jupyter kernel rendering millions of points, we leverage DuckDB's intelligence to aggregate data **before** passing it to Plotly, or utilize native sampling (`USING SAMPLE`).

In [ ]:
# Mapping refunds and financial failure volumes
query_revenue_health = f"""
    SELECT 
        CASE 
            WHEN total_amount < 0 THEN 'Negative (Refund/Error)'
            WHEN total_amount = 0 THEN 'Zero'
            ELSE 'Healthy (Positive)'
        END AS financial_integrity,
        COUNT(*) AS transaction_volume
    FROM '{RAW_FILE}'
    GROUP BY 1
    ORDER BY transaction_volume DESC
"""

df_revenue = con.sql(query_revenue_health).df()

fig = px.bar(
    df_revenue, 
    x='financial_integrity', 
    y='transaction_volume',
    title="Evidence 1: Gross Billing Integrity",
    color='financial_integrity',
    color_discrete_map={
        'Healthy (Positive)': '#2ecc71', 
        'Zero': '#f1c40f', 
        'Negative (Refund/Error)': '#e74c3c'
    },
    template="plotly_dark",
    text_auto='.2s'
)
fig.update_layout(height=400)
fig.show()

1. The Golden Rule for the Silver Layer (clean.py)
The chart mathematically proves why raw data cannot be directly copied. It justifies implementing a mandatory Data Quality Gate in your cleansing script.

    * Action: The validate_domains() function in clean.py must include the explicit constraint WHERE total_amount > 0. This acts as the sanitary barrier preventing 15,530 corrupted records from entering the clean layer.

2. Protecting Analytical Aggregations (Gold Layer)
If negative records reached transform.py queries directly, they would trigger a chain reaction damaging KPIs:

    * Revenue Distortion: SUM(total_amount) queries would subtract money from company revenue whenever encountering these negative records, resulting in a lower P&L than reality.

    * Average Ticket Distortion: AVG(total_amount) would be artificially pulled down (negative skew), penalizing driver performance metrics.

3. Separating Business Contexts
Reading this chart demonstrates analytical maturity: negative data is not necessarily "junk"; it represents a business event (cancellation). For revenue and occupancy analysis, it must be filtered. In enterprise settings, these 15k records could be isolated into a deviation table (e.g., mart_refunds_and_disputes) for Fraud Prevention teams.

In [ ]:
# Crossing Distance vs Price and translating payment method dictionary
query_scatter = f"""
    SELECT 
        trip_distance, 
        total_amount, 
        CASE payment_type
            WHEN 1 THEN 'Credit card'
            WHEN 2 THEN 'Cash'
            WHEN 3 THEN 'No charge'
            WHEN 4 THEN 'Dispute'
            WHEN 5 THEN 'Unknown'
            WHEN 6 THEN 'Voided trip'
            ELSE 'Null/Unregistered'
        END AS payment_method
    FROM '{RAW_FILE}'
    USING SAMPLE 10000
"""

df_scatter = con.sql(query_scatter).df()

fig = px.scatter(
    df_scatter, 
    x='trip_distance', 
    y='total_amount', 
    color='payment_method',
    title="Evidence 2: Disconnect between GPS (Distance) and Transaction (Price)",
    labels={
        'trip_distance': 'Distance (Miles)', 
        'total_amount': 'Total Amount ($)',
        'payment_method': 'Payment Method'
    },
    template="plotly_dark"
)

# Locking axes to reveal the true data cloud and refunds
fig.update_xaxes(range=[-2, 50])
fig.update_yaxes(range=[-50, 250])
fig.update_layout(height=450)
fig.show()

1. The NULLIF Rule (Mathematical Protection in Gold Layer)
Calculating Revenue per Mile (total_amount / trip_distance) in transform.py on raw data causes an immediate Division by Zero error due to zero-distance trips.

    * Action: Defensive math functions become mandatory (e.g., total_amount / NULLIF(trip_distance, 0)), ensuring GPS failures produce controlled nulls rather than halting pipelines.

2. Spatial Gatekeeping (Silver Layer Filtering)
If zero-distance high-fare trips reach the Gold layer, overall average ticket and fleet profitability metrics will be corrupted by false telemetry.

    * Action: The clean.py module must enforce WHERE trip_distance > 0. Stationary trips charging hundreds of dollars are analytical noise for mobility KPIs.

3. Isolating Disputes
The plot shows payment_type = 4 (Dispute) is the primary driver of negative amounts.

    * Action: Confirms the need for WHERE total_amount > 0 in clean.py to purify financial reporting while enabling potential isolation into dedicated fraud Data Marts.

In [ ]:
# Grouping nulls and translating provider IDs
query_nulls = f"""
    SELECT 
        CASE VendorID
            WHEN 1 THEN 'Creative Mobile Technologies'
            WHEN 2 THEN 'VeriFone Inc.'
            ELSE 'Unknown Provider'
        END AS technology_provider,
        COUNT(*) as total_trips,
        SUM(CASE WHEN passenger_count IS NULL THEN 1 ELSE 0 END) AS failed_trips,
        ROUND(SUM(CASE WHEN passenger_count IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS failure_rate_pct
    FROM '{RAW_FILE}'
    GROUP BY 1
"""

df_nulls = con.sql(query_nulls).df()

fig = px.bar(
    df_nulls,
    x='technology_provider',
    y='failure_rate_pct',
    title="Evidence 3: Tracking Integration Gaps by Provider",
    labels={
        'technology_provider': 'Technology Provider (Vendor)', 
        'failure_rate_pct': '% Lost Data'
    },
    template="plotly_dark",
    color='technology_provider',
    text_auto='.2f'
)
fig.update_layout(height=400, showlegend=False)
fig.show()

1. Primary Contributor (VeriFone Inc. as Weakest Link)
Nearly 23% of trips processed by VeriFone hardware/software lose vital attributes (passenger_count, RatecodeID, etc.) before arriving at TLC databases.

2. Illusion of Stability
Creative Mobile Technologies (CMT) performs better but still fails in 13.33% of transmissions, showing overall mobile telemetry instability across NYC taxis.

3. The Unknown Provider Issue
16.7% failure rate exists where VendorID itself is missing, indicating outdated hardware or legacy integrations.

In [ ]:
# Isolating zero-passenger operations
query_ghost_trips = f"""
    SELECT 
        passenger_count,
        COUNT(*) AS volume_trips,
        ROUND(AVG(total_amount), 2) AS ghost_average_ticket
    FROM '{RAW_FILE}'
    WHERE passenger_count IS NOT NULL
    GROUP BY 1
    ORDER BY 1
"""

df_ghost = con.sql(query_ghost_trips).df()
df_ghost['passenger_count'] = df_ghost['passenger_count'].astype(str)

fig = px.bar(
    df_ghost,
    x='passenger_count',
    y='volume_trips',
    title="Evidence 4: Trip Volume by Passenger Count",
    color='ghost_average_ticket',
    labels={'passenger_count': 'Passengers', 'volume_trips': 'Volume'},
    template="plotly_dark",
    text_auto='.2s'
)
fig.update_layout(height=400)
fig.show()

1. Operational Norm (Decay Curve)
Single passengers represent the vast majority (2.5M), dropping off steeply for 2 to 4 passengers. 5-6 passenger limits map to TLC SUV/minivan fleets.

2. 12k Ghost Operations (Zero Passengers)
12,000 trips registered 0 passengers while collecting fares, pointing to driver entry omission, seat sensor failure, or parcel delivery usage.

3. Physical Outliers (7-9 Passengers)
Low-volume trips with up to 9 passengers reflect manual entry errors on vehicle terminals.

🚀 Action Plan Constraints
1. Defensive Imputation (Silver Layer): Impute 0 and NULL passenger counts to 1 (statistical mode) rather than deleting paid trips.
2. Physical Boundaries: Restrict passenger count between 1 and 6 in validate_domains().
3. Mathematical Shielding (Gold Layer): Use NULLIF to protect ratio calculations against zero-division risks.